In [ ]:
# Edit only attached Kaggle Input paths and the declared offline revision if known.
from pathlib import Path

BM25_WHEEL_PATH = Path(
    "/kaggle/input/datasets/mduy2911/offline-packages/bm25s-0.3.11-py3-none-any.whl"
)
LEGALIR_SOURCE_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/train.json")
CORPUS_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/selected-contexts")
RERANKER_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/bge-reranker-v2-m3-kaggle")

RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RERANKER_DECLARED_REVISION = None  # Optional exact revision supplied with the snapshot.

CHUNK_SIZE = 2_000
CHUNK_OVERLAP = 200
BM25_METHOD = "lucene"
BM25_K1 = 1.5
BM25_B = 0.75
TOP_K_CHUNKS = 2_000
DOCUMENT_AGGREGATION = "sum_top_2"
CANDIDATE_DEPTH = 100
SUPPORTING_CHUNKS_PER_DOCUMENT = 2
FINAL_K_VALUES = (1, 2, 3, 4, 5)
RERANKER_BATCH_SIZE = 128
MAX_SEQUENCE_LENGTH_CAP = 8_192
EXPECTED_DEV_QUERIES = 1_036
BOOTSTRAP_SEED = 20_260_913
BOOTSTRAP_RESAMPLES = 10_000
RESULT_PATH = Path("/kaggle/working/cross_encoder_final_selection_dev_results.json")


In [ ]:
# Fail before imports/model loading if an offline artifact is missing.
import os
import subprocess
import sys

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

if not BM25_WHEEL_PATH.is_file():
    raise FileNotFoundError(f"Attach the local BM25 wheel at: {BM25_WHEEL_PATH}")
if not LEGALIR_SOURCE_PATH.is_file():
    raise FileNotFoundError(f"Attach LegalIR train.json at: {LEGALIR_SOURCE_PATH}")
if not CORPUS_PATH.is_dir():
    raise FileNotFoundError(f"Attach the LegalIR corpus directory at: {CORPUS_PATH}")
if not RERANKER_MODEL_PATH.is_dir():
    raise FileNotFoundError(
        f"Attach the complete local reranker snapshot at: {RERANKER_MODEL_PATH}"
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--no-deps",
        str(BM25_WHEEL_PATH),
    ],
    check=True,
)


In [ ]:
import json
import re
from collections import Counter, defaultdict
from hashlib import sha256
from math import isfinite
from time import perf_counter

import bm25s
import numpy as np
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def read_json(path: Path):
    try:
        with path.open(encoding="utf-8-sig") as stream:
            value = json.load(stream)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{path}: invalid JSON: {exc}") from exc
    return value


def load_legalir(path: Path) -> dict:
    value = read_json(path)
    if not isinstance(value, dict) or not all(
        isinstance(sample, dict) for sample in value.values()
    ):
        raise ValueError(f"{path}: expected an object keyed by sample ID")
    canonical = {str(sample_id): sample for sample_id, sample in value.items()}
    if len(canonical) != len(value):
        raise ValueError(f"{path}: duplicate sample IDs after string canonicalization")
    return canonical


def select_fixed_dev(samples: dict) -> dict:
    dev_ids = []
    for sample_id, sample in samples.items():
        question = sample.get("question")
        group_key = (
            question
            if isinstance(question, str)
            else f"\0fallback-sample-id:{sample_id}"
        )
        bucket = int(sha256(group_key.encode("utf-8")).hexdigest()[:8], 16) % 100
        if 70 <= bucket < 85:
            dev_ids.append(sample_id)
    dev_ids.sort()
    dev = {sample_id: samples[sample_id] for sample_id in dev_ids}
    if len(dev) != EXPECTED_DEV_QUERIES:
        raise ValueError(
            f"fixed DEV must contain {EXPECTED_DEV_QUERIES} queries, got {len(dev)}"
        )
    for sample_id, sample in dev.items():
        if not isinstance(sample.get("question"), str):
            raise TypeError(f"DEV sample {sample_id!r}: question must be a string")
        answer = sample.get("answer")
        if not isinstance(answer, list) or not answer:
            raise ValueError(
                f"DEV sample {sample_id!r}: expected a non-empty LegalIR answer list"
            )
    return dev


def load_corpus(path: Path) -> list[dict]:
    json_paths = sorted(
        item for item in path.rglob("*") if item.is_file() and item.suffix.lower() == ".json"
    )
    if not json_paths:
        raise ValueError(f"{path}: corpus directory contains no JSON files")
    documents = []
    for json_path in json_paths:
        value = read_json(json_path)
        values = value if isinstance(value, list) else [value]
        if not all(isinstance(document, dict) for document in values):
            raise ValueError(f"{json_path}: expected document object(s)")
        documents.extend(values)
    canonical_ids = [str(document.get("id")) for document in documents]
    if len(canonical_ids) != len(set(canonical_ids)):
        raise ValueError("corpus contains duplicate document IDs")
    return documents


def chunk_corpus(documents: list[dict]) -> list[dict]:
    if CHUNK_SIZE <= 0 or CHUNK_OVERLAP < 0 or CHUNK_OVERLAP >= CHUNK_SIZE:
        raise ValueError("invalid fixed-window chunk parameters")
    step = CHUNK_SIZE - CHUNK_OVERLAP
    chunks = []
    for document in documents:
        document_id = str(document["id"])
        passage = document.get("passage")
        if not isinstance(passage, str):
            raise TypeError(f"document {document_id!r}: passage must be a string")
        if not passage:
            continue
        for chunk_index, start in enumerate(range(0, len(passage), step)):
            end = min(start + CHUNK_SIZE, len(passage))
            chunks.append(
                {
                    "chunk_id": f"{document_id}:{chunk_index}",
                    "document_id": document_id,
                    "text": passage[start:end],
                }
            )
            if end == len(passage):
                break
    if not chunks:
        raise ValueError("fixed-window corpus produced no chunks")
    return chunks


TOKEN_PATTERN = re.compile(r"\w+", flags=re.UNICODE)


def lexical_tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.lower())


def build_bm25(chunks: list[dict]):
    if bm25s.__version__ != "0.3.11":
        raise RuntimeError(f"expected bm25s==0.3.11, got {bm25s.__version__}")
    tokenized = bm25s.tokenize(
        [chunk["text"] for chunk in chunks],
        lower=True,
        token_pattern=r"(?u)\w+",
        stopwords=[],
        stemmer=None,
        return_ids=True,
        show_progress=False,
    )
    retriever = bm25s.BM25(k1=BM25_K1, b=BM25_B, method=BM25_METHOD)
    retriever.index(tokenized, show_progress=False)
    return retriever


def aggregate_sum_top_2(chunk_hits: list[dict]) -> list[dict]:
    grouped = defaultdict(list)
    best_rank = {}
    for hit in chunk_hits:
        document_id = hit["document_id"]
        score = float(hit["score"])
        if not isfinite(score):
            raise ValueError("BM25 score must be finite")
        grouped[document_id].append(score)
        best_rank[document_id] = min(best_rank.get(document_id, hit["rank"]), hit["rank"])
    ranked = [
        {
            "document_id": document_id,
            "score": sum(sorted(scores, reverse=True)[:2]),
            "best_chunk_rank": best_rank[document_id],
        }
        for document_id, scores in grouped.items()
    ]
    ranked.sort(
        key=lambda item: (-item["score"], item["best_chunk_rank"], item["document_id"])
    )
    return ranked


def retrieve_fixed_candidates(retriever, chunks: list[dict], samples: dict) -> dict:
    sample_items = list(samples.items())
    candidates_by_query = {}
    started = perf_counter()
    for batch_start in range(0, len(sample_items), 64):
        batch = sample_items[batch_start : batch_start + 64]
        result = retriever.retrieve(
            [lexical_tokenize(sample["question"]) for _, sample in batch],
            k=TOP_K_CHUNKS,
            sorted=True,
            return_as="tuple",
            show_progress=False,
        )
        for (sample_id, _), hit_indices, hit_scores in zip(
            batch, result.documents, result.scores
        ):
            chunk_hits = []
            for rank, (index_value, score_value) in enumerate(
                zip(hit_indices, hit_scores), start=1
            ):
                chunk = chunks[int(index_value)]
                chunk_hits.append(
                    {
                        "chunk_index": int(index_value),
                        "document_id": chunk["document_id"],
                        "score": float(score_value),
                        "rank": rank,
                    }
                )
            selected = aggregate_sum_top_2(chunk_hits)[:CANDIDATE_DEPTH]
            if len(selected) != CANDIDATE_DEPTH:
                raise RuntimeError(
                    f"sample {sample_id!r}: expected {CANDIDATE_DEPTH} candidates"
                )
            support = {item["document_id"]: [] for item in selected}
            for hit in chunk_hits:
                values = support.get(hit["document_id"])
                if values is not None and len(values) < SUPPORTING_CHUNKS_PER_DOCUMENT:
                    chunk = chunks[hit["chunk_index"]]
                    values.append(
                        {
                            "chunk_id": chunk["chunk_id"],
                            "text": chunk["text"],
                            "bm25_rank": hit["rank"],
                            "bm25_score": hit["score"],
                        }
                    )
            candidates = []
            for original_rank, item in enumerate(selected, start=1):
                supporting_chunks = support[item["document_id"]]
                if not 1 <= len(supporting_chunks) <= 2:
                    raise RuntimeError("candidate must have one or two supporting chunks")
                candidates.append(
                    {
                        "document_id": item["document_id"],
                        "original_rank": original_rank,
                        "supporting_chunks": supporting_chunks,
                    }
                )
            candidate_ids = [item["document_id"] for item in candidates]
            if len(candidate_ids) != len(set(candidate_ids)):
                raise RuntimeError(f"sample {sample_id!r}: duplicate candidate IDs")
            candidates_by_query[sample_id] = candidates
    return {
        "candidates": candidates_by_query,
        "seconds": perf_counter() - started,
    }


def local_model_metadata(model, path: Path, name: str, declared_revision):
    config_commit_hash = getattr(model.config, "_commit_hash", None)
    verified = isinstance(config_commit_hash, str) and bool(config_commit_hash.strip())
    return {
        "model_name": name,
        "local_input_path": str(path),
        "declared_revision": declared_revision,
        "config_commit_hash": config_commit_hash if verified else None,
        "revision_status": (
            "verified-from-config" if verified else "declared-offline-snapshot"
        ),
    }


def load_reranker():
    if not torch.cuda.is_available():
        raise RuntimeError("This reranking notebook requires a Kaggle CUDA accelerator")
    torch.cuda.reset_peak_memory_stats()
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(
        RERANKER_MODEL_PATH,
        local_files_only=True,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_MODEL_PATH,
        dtype=torch.float16,
        local_files_only=True,
    )
    model.to("cuda")
    model.eval()
    tokenizer_limit = int(tokenizer.model_max_length)
    model_limit = int(
        getattr(model.config, "max_position_embeddings", tokenizer_limit)
    )
    max_length = min(tokenizer_limit, model_limit, MAX_SEQUENCE_LENGTH_CAP)
    if max_length <= 0:
        raise ValueError("resolved reranker maximum length must be positive")
    return {
        "tokenizer": tokenizer,
        "model": model,
        "max_length": max_length,
        "load_seconds": perf_counter() - started,
        "metadata": local_model_metadata(
            model,
            RERANKER_MODEL_PATH,
            RERANKER_MODEL_NAME,
            RERANKER_DECLARED_REVISION,
        ),
    }


def score_pairs(reranker: dict, pairs: list[tuple[str, str]]) -> dict:
    scores = []
    started = perf_counter()
    tokenizer = reranker["tokenizer"]
    model = reranker["model"]
    for batch_start in range(0, len(pairs), RERANKER_BATCH_SIZE):
        batch = pairs[batch_start : batch_start + RERANKER_BATCH_SIZE]
        encoded = tokenizer(
            [question for question, _ in batch],
            [passage for _, passage in batch],
            padding=True,
            truncation="only_second",
            max_length=reranker["max_length"],
            return_tensors="pt",
        )
        encoded = {name: value.to("cuda") for name, value in encoded.items()}
        with torch.no_grad():
            logits = model(**encoded, return_dict=True).logits.view(-1).float()
        batch_scores = logits.cpu().tolist()
        if len(batch_scores) != len(batch) or not all(
            isfinite(value) for value in batch_scores
        ):
            raise RuntimeError("reranker returned invalid scores")
        scores.extend(float(value) for value in batch_scores)
    return {"scores": scores, "seconds": perf_counter() - started}


def rerank_once(reranker: dict, samples: dict, candidates_by_query: dict) -> dict:
    pairs = []
    layout = []
    for sample_id, sample in samples.items():
        candidates = candidates_by_query[sample_id]
        counts = []
        for candidate in candidates:
            counts.append(len(candidate["supporting_chunks"]))
            pairs.extend(
                (sample["question"], supporting["text"])
                for supporting in candidate["supporting_chunks"]
            )
        layout.append((sample_id, counts))

    scoring = score_pairs(reranker, pairs)
    offset = 0
    rankings = {}
    for sample_id, counts in layout:
        scored_documents = []
        candidates = candidates_by_query[sample_id]
        for candidate, count in zip(candidates, counts):
            chunk_scores = scoring["scores"][offset : offset + count]
            offset += count
            scored_documents.append(
                {
                    "document_id": candidate["document_id"],
                    "cross_encoder_score": sum(chunk_scores),
                    "original_rank": candidate["original_rank"],
                }
            )
        scored_documents.sort(
            key=lambda item: (
                -item["cross_encoder_score"],
                item["original_rank"],
                item["document_id"],
            )
        )
        ranking = [item["document_id"] for item in scored_documents]
        candidate_ids = [item["document_id"] for item in candidates]
        if len(ranking) != len(set(ranking)) or set(ranking) != set(candidate_ids):
            raise RuntimeError("reranking changed the fixed candidate universe")
        rankings[sample_id] = ranking
    if offset != len(scoring["scores"]):
        raise RuntimeError("not every cross-encoder score was consumed")
    return {
        "rankings": rankings,
        "number_of_pairs": len(pairs),
        "seconds": scoring["seconds"],
    }


def bundled_eval_retrieval(predictions: dict, truth: dict) -> dict:
    # Inline equivalent of scoring/LegalIR/scoring.py eval_retrieval.
    predicted = {sample_id: value["answer"] for sample_id, value in predictions.items()}
    if len(predicted) != len(truth):
        raise ValueError("samples in predictions do not match the reference")
    recall = np.asarray([
        len(set(truth[sample_id]) & set(predicted.get(sample_id, set())))
        / len(truth[sample_id])
        if 0 < len(predicted.get(sample_id, [])) <= 5 else 0.0
        for sample_id in truth
    ]).mean()
    precision = np.asarray([
        len(set(truth[sample_id]) & set(predicted.get(sample_id, set())))
        / len(predicted[sample_id])
        if 0 < len(predicted.get(sample_id, [])) <= 5 else 0.0
        for sample_id in predicted
    ]).mean()
    return {"precision": float(precision), "recall": float(recall)}


def bundled_contributions(samples: dict, rankings: dict, k: int) -> dict:
    precision = []
    recall = []
    any_correct = 0
    all_recovered = 0
    lengths = []
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        prediction = [str(document_id) for document_id in rankings[sample_id][:k]]
        if not 0 < len(prediction) <= 5:
            overlap = 0
            precision_value = 0.0
            recall_value = 0.0
        else:
            if len(prediction) != len(set(prediction)):
                raise ValueError(f"sample {sample_id!r}: duplicate predicted IDs")
            overlap = len(gold.intersection(prediction))
            precision_value = overlap / len(prediction)
            recall_value = overlap / len(gold)
        precision.append(precision_value)
        recall.append(recall_value)
        lengths.append(len(prediction))
        any_correct += overlap > 0
        all_recovered += overlap == len(gold)
    predictions = {
        sample_id: {"answer": rankings[sample_id][:k]}
        for sample_id in samples
    }
    truth = {sample_id: sample["answer"] for sample_id, sample in samples.items()}
    bundled = bundled_eval_retrieval(predictions, truth)
    if not np.isclose(bundled["precision"], np.mean(precision)):
        raise RuntimeError("precision contribution mean differs from bundled scorer")
    if not np.isclose(bundled["recall"], np.mean(recall)):
        raise RuntimeError("recall contribution mean differs from bundled scorer")
    return {
        "precision_values": precision,
        "recall_values": recall,
        "precision": bundled["precision"],
        "recall": bundled["recall"],
        "average_prediction_length": float(np.mean(lengths)),
        "queries_with_at_least_one_correct_output": int(any_correct),
        "queries_with_all_gold_documents_recovered": int(all_recovered),
    }


def paired_behavior(reference: dict, alternative: dict) -> dict:
    output = {}
    for metric in ("precision", "recall"):
        counts = Counter()
        for base, candidate in zip(
            reference[f"{metric}_values"], alternative[f"{metric}_values"]
        ):
            delta = candidate - base
            counts[
                "improved" if delta > 0 else "worsened" if delta < 0 else "unchanged"
            ] += 1
        output[metric] = {
            label: counts[label] for label in ("improved", "unchanged", "worsened")
        }
    return output


def paired_bootstrap(reference: list[float], alternative: list[float]) -> dict:
    paired_deltas = np.asarray(alternative) - np.asarray(reference)
    generator = np.random.default_rng(BOOTSTRAP_SEED)
    means = np.empty(BOOTSTRAP_RESAMPLES, dtype=np.float64)
    for index in range(BOOTSTRAP_RESAMPLES):
        sampled = generator.integers(0, paired_deltas.size, paired_deltas.size)
        means[index] = paired_deltas[sampled].mean()
    return {
        "observed_delta": float(paired_deltas.mean()),
        "percentile_interval_95": [
            float(np.percentile(means, 2.5)),
            float(np.percentile(means, 97.5)),
        ],
        "seed": BOOTSTRAP_SEED,
        "resamples": BOOTSTRAP_RESAMPLES,
    }


In [ ]:
# Local preflight and synthetic model smoke check; no DEV scores are produced here.
assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
assert FINAL_K_VALUES == (1, 2, 3, 4, 5)
assert TOP_K_CHUNKS == 2_000
assert DOCUMENT_AGGREGATION == "sum_top_2"
assert CANDIDATE_DEPTH == 100
assert SUPPORTING_CHUNKS_PER_DOCUMENT == 2

reranker = load_reranker()
smoke = score_pairs(
    reranker,
    [("Câu hỏi kiểm tra.", "Đoạn văn kiểm tra.")],
)
assert len(smoke["scores"]) == 1
assert isfinite(smoke["scores"][0])
print("Offline preflight and reranker smoke check passed.")


In [ ]:
# Build the expensive fixed ranking exactly once.
run_started = perf_counter()
source_samples = load_legalir(LEGALIR_SOURCE_PATH)
dev_samples = select_fixed_dev(source_samples)
documents = load_corpus(CORPUS_PATH)
chunks = chunk_corpus(documents)
if len(documents) != 8_532 or len(chunks) != 199_816:
    raise ValueError(
        f"expected 8,532 documents / 199,816 chunks, got "
        f"{len(documents):,} / {len(chunks):,}"
    )

bm25_started = perf_counter()
retriever = build_bm25(chunks)
bm25_build_seconds = perf_counter() - bm25_started
fixed_candidates = retrieve_fixed_candidates(retriever, chunks, dev_samples)
fixed_reranked = rerank_once(
    reranker,
    dev_samples,
    fixed_candidates["candidates"],
)
reranked_rankings = fixed_reranked["rankings"]  # The only final ranking used below.

assert set(reranked_rankings) == set(dev_samples)
assert all(
    len(ranking) == CANDIDATE_DEPTH and len(ranking) == len(set(ranking))
    for ranking in reranked_rankings.values()
)


In [ ]:
# Evaluate k=1..5 only by slicing prefixes from the same reranked ranking.
per_k_internal = {
    k: bundled_contributions(dev_samples, reranked_rankings, k)
    for k in FINAL_K_VALUES
}
reference = per_k_internal[5]
comparison_table = []
paired = {}
pareto_improvements = []

for k in FINAL_K_VALUES:
    metrics = per_k_internal[k]
    precision_delta = metrics["precision"] - reference["precision"]
    recall_delta = metrics["recall"] - reference["recall"]
    is_pareto = (
        k < 5
        and metrics["precision"] >= reference["precision"]
        and metrics["recall"] >= reference["recall"]
        and (precision_delta > 0 or recall_delta > 0)
    )
    if is_pareto:
        pareto_improvements.append(k)
    comparison_table.append(
        {
            "k": k,
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "delta_precision_vs_k_5": precision_delta,
            "delta_recall_vs_k_5": recall_delta,
        }
    )
    paired[str(k)] = paired_behavior(reference, metrics)

bootstrap = None
if pareto_improvements:
    bootstrap = {
        str(k): {
            metric: paired_bootstrap(
                reference[f"{metric}_values"],
                per_k_internal[k][f"{metric}_values"],
            )
            for metric in ("precision", "recall")
        }
        for k in pareto_improvements
    }

interpretation = []
for row in comparison_table:
    if row["k"] == 5:
        continue
    if row["k"] in pareto_improvements:
        label = "pareto_improvement"
    elif row["delta_precision_vs_k_5"] > 0 and row["delta_recall_vs_k_5"] < 0:
        label = "precision_recall_trade_off"
    else:
        label = "no_automatic_selection"
    interpretation.append({"k": row["k"], "classification": label})

result = {
    "split": {
        "name": "fixed DEV",
        "queries": len(dev_samples),
        "source_sha256": sha256(LEGALIR_SOURCE_PATH.read_bytes()).hexdigest(),
    },
    "controls": {
        "documents": len(documents),
        "chunks": len(chunks),
        "chunk_size": CHUNK_SIZE,
        "overlap": CHUNK_OVERLAP,
        "bm25": {
            "library": f"bm25s=={bm25s.__version__}",
            "method": BM25_METHOD,
            "k1": BM25_K1,
            "b": BM25_B,
            "tokenization": r"lowercase Unicode \w+",
            "top_k_chunks": TOP_K_CHUNKS,
        },
        "document_aggregation": "sum top-2 BM25 chunk scores",
        "candidate_depth": CANDIDATE_DEPTH,
        "reranker": {
            **reranker["metadata"],
            "supporting_chunks_per_document": SUPPORTING_CHUNKS_PER_DOCUMENT,
            "document_score": "sum up to 2 independent cross-encoder chunk scores",
            "max_sequence_length": reranker["max_length"],
            "batch_size": RERANKER_BATCH_SIZE,
            "dtype": "float16",
        },
        "ranking_construction_count": 1,
        "evaluated_prefixes": list(FINAL_K_VALUES),
    },
    "per_k": {
        str(k): {
            key: value
            for key, value in metrics.items()
            if not key.endswith("_values")
        }
        for k, metrics in per_k_internal.items()
    },
    "comparison_table": comparison_table,
    "paired_behavior_vs_k_5": paired,
    "pareto_improvement_k": pareto_improvements,
    "interpretation": interpretation,
    "paired_bootstrap": bootstrap,
    "runtime": {
        "model_load_seconds": reranker["load_seconds"],
        "bm25_build_seconds": bm25_build_seconds,
        "candidate_retrieval_seconds": fixed_candidates["seconds"],
        "reranking_seconds": fixed_reranked["seconds"],
        "cross_encoder_pairs": fixed_reranked["number_of_pairs"],
        "peak_gpu_memory_bytes": int(torch.cuda.max_memory_allocated()),
        "total_seconds": perf_counter() - run_started,
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
    },
}

RESULT_PATH.write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("k | precision | recall | delta precision vs k=5 | delta recall vs k=5")
for row in comparison_table:
    print(
        f"{row['k']} | {row['precision']:.12f} | {row['recall']:.12f} | "
        f"{row['delta_precision_vs_k_5']:+.12f} | "
        f"{row['delta_recall_vs_k_5']:+.12f}"
    )
print(json.dumps({"interpretation": interpretation, "bootstrap": bootstrap}, indent=2))
print("Saved:", RESULT_PATH)
